# Quiz 3

### Imports

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as smf

### Datasets

In [10]:
df_1 = pd.read_csv('./datasets/homework_3.1.csv')
df_2_a = pd.read_csv('./datasets/homework_3.2.a.csv')
df_2_b = pd.read_csv('./datasets/homework_3.2.b.csv')

### Q1

### Pipeline

In [11]:
def createRegressionSplit(df, split, val):
    df_split_1 = df[df['time'] < split]
    df_split_2 = df[df['time'] > split]

    model_1 = LinearRegression()
    model_1.fit(df_split_1['time'].values.reshape(-1, 1), df_split_1[val])

    model_2 = LinearRegression()
    model_2.fit(df_split_2['time'].values.reshape(-1, 1), df_split_2[val])

    return (
        model_1.coef_,
        model_1.intercept_,
        model_2.coef_,
        model_2.intercept_
    )

In [12]:
vals = ['value1', 'value2', 'value3']

for val in vals:
    coef_1, intercept_1, coef_2, intercept_2 = createRegressionSplit(df_1, 50, val)
    gap = abs((coef_2[0] * 50 + intercept_2) - (coef_1[0] * 50 + intercept_1))
    print(f'{val}: {gap:.4f}')

value1: 1.0049
value2: 0.7668
value3: 1.8697


### Q2

In [13]:
vals = ['value1', 'value2', 'value3']

for val in vals:
    coef_1, intercept_1, coef_2, intercept_2 = createRegressionSplit(df_1, 50, val)
    gap = abs(coef_2[0] - coef_1[0])
    print(f'{val}: {gap:.4f}')

value1: 0.1023
value2: 0.0349
value3: 0.0484


### Q3

In [14]:
df_2_a.head()

,group1,time1,outcome1
0,0,0,0.882026
1,0,1,1.600079
2,0,0,0.489369
3,0,1,2.520447
4,0,0,0.933779


In [15]:
np.random.seed(42)

df_2_a['treated_after'] = df_2_a['group1'] * df_2_a['time1']
df_2_b['treated_after'] = df_2_b['group2'] * df_2_b['time2']

model_a = smf.ols(
    "outcome1 ~ group1 + time1 + treated_after",
    data=df_2_a
).fit(cov_type="HC1")

model_b = smf.ols(
    "outcome2 ~ group2 + time2 + treated_after",
    data=df_2_b
).fit(cov_type="HC1")

#print("GROUP 1")
#print(model_a.summary())

#print("\n\n\nGROUP 2")
print(model_b.summary())

                            OLS Regression Results                            
Dep. Variable:               outcome2   R-squared:                       0.663
Model:                            OLS   Adj. R-squared:                  0.662
Method:                 Least Squares   F-statistic:                     620.2
Date:                Sun, 21 Jun 2026   Prob (F-statistic):          2.62e-227
Time:                        18:27:33   Log-Likelihood:                -1567.5
No. Observations:                1000   AIC:                             3143.
Df Residuals:                     996   BIC:                             3163.
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         0.1021      0.076      1.352

### Q4

In [16]:
df_2_a['treated_after'] = df_2_a['group1'] * df_2_a['time1']
df_2_b['treated_after'] = df_2_b['group2'] * df_2_b['time2']

model_a = smf.ols(
    "outcome1 ~ group1 + time1 + treated_after",
    data=df_2_a
).fit(cov_type="HC1")

model_b = smf.ols(
    "outcome2 ~ group2 + time2 + treated_after",
    data=df_2_b
).fit(cov_type="HC1")

for name, model in [("GROUP 1", model_a), ("GROUP 2", model_b)]:
    coef = model.params['treated_after']
    pval = model.pvalues['treated_after']
    ci = model.conf_int().loc['treated_after']
    tstat = model.tvalues['treated_after']
    print(f"{name}: treated_after = {coef:.4f}, tstat = {tstat:.4f}")

GROUP 1: treated_after = 0.6858, tstat = 10.9713
GROUP 2: treated_after = 1.3499, tstat = 9.1807
